# Taller: monta tú un modelo de marketing mix

Segunda parte del seminario. En la primera hora has visto qué es una priori, qué es una
posteriori y por qué un modelo jerárquico presta información entre grupos. Aquí no vas a
repetir aquello con otros datos: vas a **construir un modelo desde cero**.

El recorrido:

1. Los datos.
2. El modelo que ajusta todo el mundo (y por qué miente).
3. Escalado.
4. Adstock y saturación: parámetros que no son betas.
5. Prioris. La parte larga.
6. Simular desde la priori, antes de mirar los datos.
7. Ajuste.
8. Diagnóstico.
9. Resultados.
10. La pregunta que le importa a alguien.

---

**Esta es la versión resuelta.** El enunciado es el mismo que el de `taller.ipynb`; lo que
cambia es que las celdas de los ejercicios vienen con código y con la respuesta escrita
debajo. Hay nueve ejercicios, uno por sección en la que hay algo que decidir.

Dos avisos. Los números concretos salen de `SEMILLA = 42`: si la cambias, las cifras de las
respuestas se mueven en el último decimal, no las conclusiones. Y si no quieres esperar al
muestreo, en la sección 7 hay un `MUESTREAR = False` que carga la posteriori guardada.


In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pymc as pm
import pytensor.tensor as pt
import arviz as az
import statsmodels.api as sm

from src.estilo import ACENTO, GRISES, aplicar_estilo
from src.mmm import (
    CANALES,
    CONTROLES,
    MAX_LAG,
    cargar_datos,
    escalar_controles,
    escalar_kpi,
    escalar_medios,
    hill,
    matriz_rezagos,
    pesos_adstock,
    vida_media,
)

aplicar_estilo()
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos

Un *marketing mix model* responde a una pregunta vieja: de todo lo que vendo, ¿cuánto se
debe a lo que me gasto en publicidad? Antes se hacía con una regresión y una hoja de
cálculo. Ahora se hace con una regresión y un modelo bayesiano, que es lo mismo pero
admitiendo en voz alta que los datos no dan para tanto.

Usamos los datos simulados de **Google Meridian**, la librería de MMM de Google. 156
semanas, cinco canales de pago con impresiones y gasto, dos controles y un KPI.

Meridian presume, con razón, de su modelo jerárquico por regiones: en su repositorio hay
un fichero por geografías (`geo_all_channels.csv`) donde cada región tiene su propio
efecto y todas se prestan información, exactamente lo que has visto con los condados de
Minnesota. **Aquí vamos a usar el modelo nacional.** No porque el jerárquico sobre, sino
porque hoy el tema no es la jerarquía: es todo lo demás que hay que decidir para que un
modelo bayesiano sea defendible.

In [ ]:
df = cargar_datos(RAIZ)
print(f"{len(df)} semanas, de {df['time'].min():%Y-%m-%d} a {df['time'].max():%Y-%m-%d}")
df.head()

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ejes[0].plot(df["time"], df["revenue"] / 1e6, color=ACENTO)
ejes[0].set_ylabel("Ingresos (M€)")

for canal, color in zip(CANALES, GRISES + [ACENTO]):
    ejes[1].plot(df["time"], df[f"{canal}_spend"] / 1e3, color=color, label=canal, lw=1)
ejes[1].set_ylabel("Inversión (k€)")
ejes[1].legend(ncol=5, fontsize=9)

fig.tight_layout()
plt.show()

### Ejercicio 1

Antes de modelar nada, mira los datos como si te los acabaran de pasar por correo:

- ¿Cuánto se ha invertido en total en cada canal y cuánto se ha ingresado?
- ¿Hay semanas sin inversión en algún canal?
- ¿Qué correlación hay entre la inversión de cada canal y los ingresos? ¿Y entre canales?

La última pregunta es la importante: si dos canales suben y bajan a la vez, ningún modelo
del mundo va a poder separarlos limpiamente. Ni bayesiano ni de los otros.

In [ ]:
inversion = [f"{c}_spend" for c in CANALES]

print(f"ingresos: {df['revenue'].sum() / 1e6:.0f} M€ en {len(df)} semanas")
print(f"inversión: {df[inversion].sum().sum() / 1e6:.0f} M€")

resumen_canales = pd.DataFrame({
    "inversión (M€)": df[inversion].sum().to_numpy() / 1e6,
    "% del presupuesto": 100 * df[inversion].sum().to_numpy() / df[inversion].sum().sum(),
    "semanas a cero": (df[inversion] == 0).sum().to_numpy(),
    "corr. con ingresos": [df[c].corr(df["revenue"]) for c in inversion],
}, index=CANALES)
display(resumen_canales.round(2))

# La pregunta importante: ¿se mueven los canales juntos?
correlaciones = df[inversion].corr()
display(correlaciones.round(2))

triangulo = np.triu(np.ones(correlaciones.shape), k=1).astype(bool)
pares = correlaciones.where(triangulo).stack().sort_values(ascending=False)
print("los tres pares de canales más correlacionados entre sí:")
print(pares.head(3).round(2).to_string())

**Respuesta.** 1.319 M€ de ingresos en 156 semanas y 219 M€ de inversión, muy mal
repartidos: `Channel3` se lleva el 40 % del presupuesto y `Channel2` el 5,5 %, siete veces
menos. Y solo `Channel2` tiene semanas sin inversión: cuatro.

Las correlaciones con los ingresos son un desastre: +0,05 y +0,01 en `Channel0` y
`Channel1`, y **negativas** en los otros tres, hasta −0,18 en `Channel4`. Leído a lo bruto:
cuanto más se invierte, menos se factura.

Entre canales, `Channel2`–`Channel3` están a 0,70, y `Channel2`–`Channel4` y
`Channel3`–`Channel4` a 0,64 y 0,62. Esos tres suben y bajan juntos, así que ningún modelo
—bayesiano o no— va a repartir el mérito entre ellos a partir de los datos. Lo que salga de
ahí lo vas a poner tú en la priori. Más vale saberlo antes de ajustar que después.

## 2. El modelo que ajusta todo el mundo

Ingresos contra inversión de cada canal, más los dos controles. Mínimos cuadrados. El
coeficiente de cada canal se lee directamente como **ROI**: euros de ingreso incremental
por euro invertido.

In [ ]:
X = df[[f"{c}_spend" for c in CANALES] + CONTROLES]
ols = sm.OLS(df["revenue"], sm.add_constant(X)).fit()

filas = [f"{c}_spend" for c in CANALES]
resumen = pd.DataFrame({
    "roi_ols": ols.params[filas].values,
    "ic_bajo": ols.conf_int().loc[filas, 0].values,
    "ic_alto": ols.conf_int().loc[filas, 1].values,
    "p_valor": ols.pvalues[filas].values,
    "corr_con_ingresos": [df[f].corr(df["revenue"]) for f in filas],
}, index=CANALES)

print(f"R² = {ols.rsquared:.3f}")
resumen.round(2)

Un $R^2$ de 0,24 y ningún canal significativo. Los intervalos de confianza van de perder
dinero a doblarlo, todos. Y si miras las correlaciones simples, tres de los cinco canales
correlacionan **negativamente** con la facturación.

Leído en frío, este modelo dice que la publicidad no sirve para nada. También podría estar
diciendo que la empresa invierte más justo cuando las cosas van peor. Con 156 semanas y
cinco canales que suben y bajan a la vez, las dos explicaciones caben igual de bien.

### Ejercicio 2

- Mira la matriz de correlaciones entre las inversiones de los canales. ¿Qué dos canales
  no vas a poder separar nunca?
- Quita un canal del modelo y vuelve a ajustar. ¿Cuánto se mueven los coeficientes de los
  demás?
- Si tuvieras que dar un número, ¿qué ROI le pondrías al canal 2? ¿Y con qué cara?

La salida fácil es pedir más datos. No los va a haber: la inversión la decide un equipo de
marketing, no un experimento aleatorizado, y lleva años decidiéndola igual. La otra salida
es escribir en el modelo lo que ya sabes del negocio. Eso es el resto del taller.

In [ ]:
def ajustar(canales):
    """El mismo OLS de antes, con los canales que le pases."""
    X = df[[f"{c}_spend" for c in canales] + CONTROLES]
    return sm.OLS(df["revenue"], sm.add_constant(X)).fit()

coeficientes = {"completo": [ols.params[f"{c}_spend"] for c in CANALES]}
for fuera in CANALES:
    restantes = [c for c in CANALES if c != fuera]
    ajuste = ajustar(restantes)
    coeficientes[f"sin {fuera}"] = [
        ajuste.params[f"{c}_spend"] if c != fuera else np.nan for c in CANALES
    ]

tabla = pd.DataFrame(coeficientes, index=CANALES)
display(tabla.round(2))

sin_canal = tabla.drop(columns="completo")
desvio = 100 * sin_canal.sub(tabla["completo"], axis=0).abs().div(tabla["completo"].abs(), axis=0)
print("cuánto se mueve como máximo cada ROI al quitar un canal (% sobre el modelo completo):")
print(desvio.max(axis=1).round(0).to_string())

# El ROI del canal 2, con todo lo que lo acompaña.
ic = ols.conf_int().loc["Channel2_spend"]
print(f"\nChannel2: ROI {ols.params['Channel2_spend']:.2f}, "
      f"IC 95 % [{ic.iloc[0]:.2f}, {ic.iloc[1]:.2f}], p = {ols.pvalues['Channel2_spend']:.2f}")

**Respuesta.** `Channel2` y `Channel3` son los que no vas a separar: 0,70 de correlación
entre sus inversiones, el par más alto de la tabla.

Y quitar canales lo confirma. El ROI de `Channel2` pasa de 1,63 a 1,89 cuando sale
`Channel3`, y a 1,89 otra vez cuando sale `Channel4`: un 16 % de movimiento por tocar *otra*
variable. `Channel4` se mueve hasta un 22 % (de 0,73 a 0,88 al quitar `Channel2`) y
`Channel0` un 11 %. Es decir: cambiar la especificación mueve los coeficientes más de lo que
los mueve cualquier cosa que hayan hecho los datos. Con cinco canales hay 31 modelos
posibles y todos son defendibles por igual.

¿Qué ROI le pongo a `Channel2`? El modelo dice 1,63, con un intervalo de confianza de
[−0,40; 3,65] y p = 0,11. O sea: entre perder 40 céntimos por euro y ganar 2,65. Si doy el
1,63 a secas, estoy mintiendo por omisión; si doy el intervalo entero, no sirve para decidir
nada. Con esa cara.

Ese modelo, además, asume dos cosas que son falsas:

1. Que el anuncio de esta semana no vende nada la semana que viene.
2. Que el euro un millón hace lo mismo que el primero.

Arreglar las dos tiene un precio: cada arreglo mete parámetros que los datos no
identifican solos. Y ahí es donde hay que decidir prioris.

## 3. Escalado

Los ingresos están en millones y las impresiones en decenas de millones. Si metes eso tal
cual en un modelo, cualquier priori que escribas es un disparate: `Normal(0, 5)` sobre una
variable que vale 8.000.000 no es una priori poco informativa, es una priori
absurdamente informativa a favor del cero.

Meridian escala así (y lo copiamos tal cual):

- **KPI**: se le resta la media y se divide por la desviación típica.
- **Medios**: cada canal se divide por la **mediana de sus semanas con inversión**. Una
  unidad = "una semana normal de ese canal".
- **Controles**: media cero, desviación típica uno.

In [ ]:
y = df["revenue"].to_numpy()
x = df[[f"{c}_impression" for c in CANALES]].to_numpy(float)
gasto = df[[f"{c}_spend" for c in CANALES]].to_numpy(float)
z = df[CONTROLES].to_numpy(float)

y_esc, y_media, y_sd = escalar_kpi(y)
x_esc, escala_medios = escalar_medios(x)
z_esc = escalar_controles(z)

gasto_total = gasto.sum(axis=0)

pd.DataFrame({
    "escala (mediana impresiones > 0)": escala_medios,
    "inversión total (€)": gasto_total,
    "semanas a cero": (x == 0).sum(axis=0),
}, index=CANALES).round(0)

### Ejercicio 3

- ¿Por qué la mediana de los valores positivos y no la media de todo? Fíjate en el canal
  que tiene semanas a cero.
- Comprueba que `y_esc` tiene media cero y desviación típica uno.
- ¿Qué valor tiene `x_esc` en una semana típica de cada canal? ¿Y en la mejor semana?

Este paso parece fontanería. No lo es: **todas las prioris del resto del taller están
escritas en esta escala**. Si cambias el escalado, cambias el modelo.

In [ ]:
print(f"y_esc: media {y_esc.mean():+.1e}, desviación típica {y_esc.std():.3f}")

# Mediana de los positivos frente a media de todo, canal a canal.
comparacion = pd.DataFrame({
    "mediana de positivos (la que usamos)": escala_medios,
    "media de todo": x.mean(axis=0),
    "semanas a cero": (x == 0).sum(axis=0),
    "x_esc semana típica": [np.median(col[col > 0]) for col in x_esc.T],
    "x_esc mejor semana": x_esc.max(axis=0),
}, index=CANALES)
display(comparacion.round(2))

print(f"diferencia entre las dos escalas, en %:")
print((100 * (x.mean(axis=0) / escala_medios - 1)).round(0))

**Respuesta.** `y_esc` tiene media cero (−2,4e−16, que es cero con decimales de ordenador)
y desviación típica uno. Lo que se esperaba, pero se comprueba.

En la semana típica `x_esc` vale exactamente 1 en los cinco canales, por construcción, y en
la mejor semana llega a 1,8–2,7 medianas, salvo `Channel2`, que se va a 5,4. Esto importa
más de lo que parece: **el rango útil de la curva de saturación está entre 0 y 3**, no entre
0 y 10. Por eso la priori de `ec` se centra en 0,8 y no en 5.

¿Por qué la mediana de los positivos y no la media de todo? Mira `Channel2`, el de las cuatro
semanas a cero: ahí la media de todo está un 23 % por encima de la mediana de los positivos,
mientras que en los demás canales la diferencia es del 1 al 8 %. La media mezcla dos cosas
que no quieres: las semanas sin inversión, que no dicen nada sobre cuánto es "mucho" en ese
canal, y los picos de campaña, que la estiran. El resultado es que cada canal quedaría
escalado con una distorsión distinta, y las prioris dejarían de significar lo mismo en todos.
Por eso Meridian escala así.

## 4. Adstock y saturación: parámetros que no son betas

Hasta ahora todo lo que has ajustado en tu vida tenía la forma "coeficiente por variable".
Un MMM no. Antes de multiplicar por nada, la inversión pasa por dos transformaciones con
parámetros propios, que también hay que estimar.

**Adstock** (el anuncio de hoy sigue vendiendo dentro de tres semanas):

$$\tilde{x}_{t} = \frac{\sum_{l=0}^{L} \alpha^{l}\, x_{t-l}}{\sum_{l=0}^{L} \alpha^{l}}, \qquad \alpha \in [0, 1]$$

Los pesos se normalizan a propósito: $\alpha$ **reparte** el efecto en el tiempo, no lo
infla. Meridian usa $L = 8$ semanas.

**Saturación de Hill** (el euro un millón rinde menos que el primero):

$$f(\tilde{x}) = \frac{\tilde{x}}{\tilde{x} + ec}$$

Va de 0 a 1 y vale exactamente 0,5 cuando $\tilde{x} = ec$: **$ec$ es el punto de media
saturación**, medido en medianas del canal. Meridian deja fijo el exponente (`slope = 1`),
que es lo que fuerza la curva a ser cóncava.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))

lags = np.arange(MAX_LAG + 1)
for alpha, color in zip([0.1, 0.4, 0.7, 0.9], GRISES[:3] + [ACENTO]):
    w = pesos_adstock(np.array([alpha]), MAX_LAG)[0]
    ejes[0].plot(lags, w, "o-", color=color,
                 label=f"α = {alpha} (vida media {vida_media(alpha):.1f} sem.)")
ejes[0].set_xlabel("semanas desde la impresión")
ejes[0].set_ylabel("peso")
ejes[0].legend(fontsize=9)

rejilla = np.linspace(0, 5, 200)
for ec, color in zip([0.3, 0.8, 2.0, 4.0], GRISES[:3] + [ACENTO]):
    ejes[1].plot(rejilla, hill(rejilla, ec), color=color, label=f"ec = {ec}")
ejes[1].set_xlabel("inversión (medianas del canal)")
ejes[1].set_ylabel("efecto relativo")
ejes[1].legend(fontsize=9)

fig.tight_layout()
plt.show()

### Ejercicio 4

- ¿Qué $\alpha$ hace falta para que el efecto de una impresión dure un mes? Usa
  `vida_media`.
- Con `ec = 0.8`, ¿qué porcentaje del efecto máximo consigue una semana normal
  ($\tilde{x} = 1$)? ¿Y una semana en la que inviertes el triple?
- Un canal de televisión y uno de búsqueda pagada, ¿tienen el mismo $\alpha$? ¿Y el mismo
  $ec$? Escribe tu respuesta antes de seguir: acabas de formular una priori.

In [ ]:
# ¿Qué alpha deja el efecto a la mitad al mes (4 semanas)? Despejando vida_media:
alpha_mes = 0.5 ** (1 / 4)
print(f"alpha = {alpha_mes:.3f}  ->  vida media {vida_media(alpha_mes):.1f} semanas")
for a in [0.1, 0.4, 0.7, 0.9]:
    print(f"alpha = {a}  ->  vida media {vida_media(a):.1f} semanas")

# Saturación con ec = 0.8: semana normal contra semana del triple.
for mult in [1.0, 2.0, 3.0]:
    print(f"\ninversión = {mult:.0f} medianas -> {100 * hill(mult, 0.8):.0f} % del efecto máximo")
print(f"\ntriplicar la inversión sube el efecto de {100 * hill(1.0, 0.8):.0f} % a "
      f"{100 * hill(3.0, 0.8):.0f} %: un {100 * (hill(3.0, 0.8) / hill(1.0, 0.8) - 1):.0f} % más "
      f"de efecto por un 200 % más de dinero")

**Respuesta.** Para que el efecto dure un mes (vida media de 4 semanas) hace falta
`alpha = 0,841`. Es mucho más alto de lo que sugiere la intuición: con el 0,4 del gráfico la
vida media es de 0,8 semanas —el efecto se ha ido antes de la semana siguiente— y hay que
llegar a 0,9 para tener 6,6 semanas de memoria.

Con `ec = 0.8`, una semana normal (`x̃ = 1`) consigue el **56 %** del efecto máximo y una
semana del triple el **79 %**. Triplicar la inversión sube el efecto un 42 %: el tercer
millón trabaja mucho peor que el primero. Eso es la concavidad, y es la razón por la que un
MMM sin saturación siempre recomienda gastar más.

¿Mismo `alpha` y mismo `ec` en televisión y en búsqueda pagada? No. La tele deja recuerdo,
así que `alpha` alto, de 0,7 a 0,9, semanas de memoria; en búsqueda la gente hace clic hoy o
no lo hace, así que `alpha` por debajo de 0,2. Y en `ec`, la búsqueda satura antes, porque el
número de personas buscando tu producto tiene techo, mientras que en tele siempre queda
alguien a quien no has llegado todavía.

Acabo de escribir todo eso sin mirar los datos. Eso es exactamente una priori. Y es el
argumento del taller: ya tenías opiniones. La diferencia es que antes se te colaban
eligiendo especificaciones hasta que el resultado te gustaba, y así no hay manera de
discutirlas.

## 5. Prioris

Ya tenemos el modelo entero salvo un detalle. Este es, escrito del todo:

$$
\begin{aligned}
y_t &\sim \mathcal{N}\!\left(\mu + \sum_m \beta_m f_{tm} + \sum_c \gamma_c z_{tc},\; \sigma^2\right) \\
f_{tm} &= \frac{\tilde{x}_{tm}}{\tilde{x}_{tm} + ec_m}, \qquad \tilde{x}_{tm} = \text{adstock}(x_{tm}; \alpha_m)
\end{aligned}
$$

Y estas son las prioris, todas sobre datos ya escalados. Tres son las de Meridian tal cual,
dos las hemos apretado y una se queda ancha a propósito:

| Parámetro | Priori | Qué dice |
|---|---|---|
| $\mu$ | $\mathcal{N}(0, 5)$ | el nivel base. La única ancha, y a propósito |
| $\alpha_m$ | $\mathcal{U}(0, 1)$ | ni idea de cuánto dura el efecto |
| $ec_m$ | $\text{TruncNormal}(0{,}8;\ 0{,}8;\ [0{,}1;\ 10])$ | la media saturación cae cerca de una semana normal |
| $\gamma_c$ | $\mathcal{N}(0, 1)$ | un control que se mueve una desviación típica mueve el KPI, como mucho, otra |
| $\sigma$ | $\text{Exp}(1)$ | el error no puede ser mayor que la desviación típica de lo que quieres explicar |

**Las dos que hemos apretado.** Meridian pone $\mathcal{N}(0, 5)$ en los controles y
$\text{HalfNormal}(5)$ en el error. Con miles de observaciones repartidas por regiones eso
da igual: los datos las tapan. Aquí no. `y_esc` tiene desviación típica 1 por construcción,
así que $\text{HalfNormal}(5)$ está diciendo que el error típico de tu modelo puede valer
cinco veces la desviación típica de lo que intentas explicar. Simulando desde esas prioris,
más de un 3 % de las semanas salen con ingresos **negativos**. Vender menos que nada.

**Y $\mu$, que sí se queda ancha.** Porque no es el nivel de los ingresos: es lo que queda
*después* de restar los medios. Con la priori del ROI que viene ahora, el término de medios
aporta de media casi cuatro desviaciones típicas, así que $\mu$ tiene que poder valer $-4$
para compensarlo. Apretarla "por simetría" es el error contrario, y ese no se ve en ningún
gráfico: se ve semanas después, cuando el muestreador va raro y nadie sabe por qué.

Falta $\beta_m$. Y aquí está el giro.

### $\beta$ no lleva priori

¿Qué priori le pondrías a $\beta_2$? Es el coeficiente de una curva de Hill sobre una
variable dividida por su mediana. Nadie tiene una intuición sobre eso. Nadie.

Sobre lo que sí tiene intuición el equipo de marketing es sobre el **ROI**: cuánto ingreso
incremental deja un euro invertido en ese canal. Así que la priori se pone ahí:

$$\text{roi}_m \sim \text{LogNormal}(0{,}2;\ 0{,}9)$$

y $\beta_m$ se **deduce**. El ingreso incremental del canal $m$ es lo que se pierde si lo
apagas del todo, y por construcción tiene que valer $\text{roi}_m$ por lo invertido:

$$\beta_m \cdot \sum_t f_{tm} \cdot sd(y) = \text{roi}_m \cdot \text{gasto}_m
\quad\Longrightarrow\quad
\beta_m = \frac{\text{roi}_m \cdot \text{gasto}_m}{sd(y) \cdot \sum_t f_{tm}}$$

$\beta_m$ deja de ser un parámetro con vida propia: es una cuenta que depende del ROI, del
adstock y de la saturación. Esto es lo mejor que hace Meridian, y no tiene nada que ver
con la programación probabilística: es reconocer que **la priori hay que escribirla en las
unidades en las que la gente sabe pensar**.

### Ejercicio 5

Traduce $\text{LogNormal}(0{,}2;\ 0{,}9)$ a algo que puedas decir en una reunión:

- ¿Cuál es el ROI mediano que asume esa priori?
- ¿Entre qué dos valores está el 90 % central?
- ¿Qué probabilidad le da a que un canal pierda dinero (ROI < 1)?

Pista: `scipy.stats.lognorm(s=0.9, scale=np.exp(0.2))`.

Y la pregunta incómoda: esa misma priori se le pone a los cinco canales. ¿Te parece
razonable, sabiendo que uno de ellos se lleva diez veces más presupuesto que otro?

In [ ]:
from scipy import stats

priori_roi = stats.lognorm(s=0.9, scale=np.exp(0.2))

print(f"ROI mediano:        {priori_roi.median():.2f}")
print(f"90 % central:       [{priori_roi.ppf(0.05):.2f}, {priori_roi.ppf(0.95):.2f}]")
print(f"P(ROI < 1):         {priori_roi.cdf(1):.0%}   (el canal pierde dinero)")
print(f"P(ROI > 5):         {priori_roi.sf(5):.0%}   (el canal quintuplica)")
print(f"media:              {priori_roi.mean():.2f}   (la cola la infla)")

rejilla = np.linspace(0, 10, 400)
fig, eje = plt.subplots(figsize=(7, 3.2))
eje.plot(rejilla, priori_roi.pdf(rejilla), color=ACENTO)
eje.fill_between(rejilla, priori_roi.pdf(rejilla), where=rejilla < 1, color="#CCCCCC",
                 label=f"ROI < 1: {priori_roi.cdf(1):.0%}")
eje.axvline(1.0, color="black", lw=0.8)
eje.set_xlabel("ROI (€ de ingreso por € invertido)")
eje.legend(fontsize=9)
plt.show()

**Respuesta.** Traducida a lenguaje de reunión: *"damos por hecho que un canal cualquiera
devuelve alrededor de 1,22 € por cada euro, que lo normal está entre 0,28 y 5,37, y que hay
un 41 % de probabilidades de que pierda dinero"*. De paso: la media es 1,83, bastante mayor
que la mediana, porque la cola de la lognormal tira. Si alguien te pregunta por el ROI
"medio" que asume tu priori, la respuesta honesta es la mediana.

Es una priori mucho menos inocente de lo que parece. Casi una de cada dos veces apuesta a
que el canal no se paga, y a la vez deja un 6 % de probabilidad a un ROI por encima de 5.
Esto es lo que quiere decir "poco informativa" en la práctica: no es neutral, es ancha.

La pregunta incómoda: no, la misma priori para los cinco canales no es razonable. Es cómoda.
Si `Channel3` se lleva el 40 % del presupuesto y lleva años llevándoselo, alguien en
marketing tiene un motivo, y ese motivo es información que el modelo no está usando. Lo
defendible sería un `mu` por canal, negociado con quien decide la inversión y escrito antes
de ver los resultados, o calibrar con un experimento (un *geo-lift*) el canal del que sí
tengas medición y dejar los demás anchos. Lo que hace el taller es el punto de partida: la
misma para todos, y que los datos muevan lo que puedan mover.

## 6. Simular desde la priori

Una priori no se juzga mirando su fórmula, se juzga mirando **qué datos genera**. Vamos a
montar el modelo y a pedirle que invente 156 semanas de ingresos sin haber visto ni uno.

In [ ]:
rezagos = matriz_rezagos(x_esc, MAX_LAG)  # (semanas, rezagos, canales)

coords = {"canal": CANALES, "control": CONTROLES, "semana": df["time"].dt.date.astype(str)}

with pm.Model(coords=coords) as modelo:
    alpha = pm.Uniform("alpha", 0.0, 1.0, dims="canal")
    ec = pm.TruncatedNormal("ec", mu=0.8, sigma=0.8, lower=0.1, upper=10.0, dims="canal")
    roi = pm.LogNormal("roi", mu=0.2, sigma=0.9, dims="canal")
    gamma = pm.Normal("gamma", 0.0, 1.0, dims="control")
    mu = pm.Normal("mu", 0.0, 5.0)  # la ancha a propósito: compensa la contribución de medios
    sigma = pm.Exponential("sigma", 1.0)

    # Adstock: media ponderada de los 9 rezagos, con pesos que suman uno.
    pesos = pesos_adstock(alpha, MAX_LAG)                 # (canales, rezagos)
    adstock = pt.sum(rezagos * pesos.T[None, :, :], axis=1)  # (semanas, canales)
    f = hill(adstock, ec)                                  # (semanas, canales)

    # Beta no es un parámetro libre: sale del ROI.
    beta = pm.Deterministic(
        "beta", roi * gasto_total / (y_sd * pt.sum(f, axis=0)), dims="canal"
    )

    contribucion = pm.Deterministic("contribucion", roi * gasto_total, dims="canal")

    media = mu + pt.dot(f, beta) + pt.dot(z_esc, gamma)
    pm.Normal("y", mu=media, sigma=sigma, observed=y_esc, dims="semana")

modelo

In [ ]:
with modelo:
    previa = pm.sample_prior_predictive(draws=500, random_seed=SEMILLA)

y_previo = previa.prior_predictive["y"].values.reshape(-1, len(df)) * y_sd + y_media

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))

ejes[0].hist(y_previo.ravel() / 1e6, bins=80, color="#AAAAAA", density=True,
             label="simulado desde la priori")
ejes[0].hist(y / 1e6, bins=30, color=ACENTO, density=True, alpha=0.8, label="observado")
ejes[0].set_xlabel("Ingresos semanales (M€)")
ejes[0].legend(fontsize=9)

contrib_previa = previa.prior["contribucion"].values.reshape(-1, len(CANALES)).sum(axis=1)
ejes[1].hist(100 * contrib_previa / y.sum(), bins=80, color=ACENTO)
ejes[1].set_xlabel("% de los ingresos atribuido a los medios, según la priori")

fig.tight_layout()
plt.show()

### Ejercicio 6

El gráfico de la izquierda es el de siempre: ¿el modelo considera normales ingresos
imposibles? El de la derecha es el que de verdad importa.

- ¿Qué porcentaje de los ingresos le atribuye la priori a la publicidad, antes de ver un
  solo dato? Calcula su mediana y su percentil 95.
- Si tu jefe te dice que la publicidad explica como mucho el 30 % de la facturación,
  ¿qué priori sobre el ROI tendrías que escribir? Pruébala y vuelve a simular.
- Vuelve a simular con las prioris originales de Meridian (`sigma = HalfNormal(5)` y
  `gamma = Normal(0, 5)`). ¿Qué porcentaje de las semanas simuladas sale con ingresos
  negativos?

Una priori que atribuye a la publicidad más ingresos de los que existen no es "poco
informativa". Es falsa. Y arreglarla ahora es gratis; después del ajuste, ya no.

In [ ]:
def simular_priori(mu_roi=0.2, sigma_roi=0.9, sigma_gamma=1.0, meridian=False, draws=500):
    """Vuelve a montar el modelo cambiando solo las prioris que nos interesan.

    Con `meridian=True` usa las originales de la librería: sigma ~ HalfNormal(5) y
    gamma ~ Normal(0, 5). El modelo de la sección 6 no se toca.
    """
    with pm.Model(coords=coords):
        alpha_ = pm.Uniform("alpha", 0.0, 1.0, dims="canal")
        ec_ = pm.TruncatedNormal("ec", mu=0.8, sigma=0.8, lower=0.1, upper=10.0, dims="canal")
        roi_ = pm.LogNormal("roi", mu=mu_roi, sigma=sigma_roi, dims="canal")
        gamma_ = pm.Normal("gamma", 0.0, 5.0 if meridian else sigma_gamma, dims="control")
        mu_ = pm.Normal("mu", 0.0, 5.0)
        sigma_ = pm.HalfNormal("sigma", 5.0) if meridian else pm.Exponential("sigma", 1.0)

        pesos_ = pesos_adstock(alpha_, MAX_LAG)
        f_ = hill(pt.sum(rezagos * pesos_.T[None, :, :], axis=1), ec_)
        beta_ = pm.Deterministic(
            "beta", roi_ * gasto_total / (y_sd * pt.sum(f_, axis=0)), dims="canal"
        )
        pm.Deterministic("contribucion", roi_ * gasto_total, dims="canal")
        media_ = mu_ + pt.dot(f_, beta_) + pt.dot(z_esc, gamma_)
        pm.Normal("y", mu=media_, sigma=sigma_, observed=y_esc, dims="semana")
        return pm.sample_prior_predictive(draws=draws, random_seed=SEMILLA)


def pct_medios(simulacion):
    """% de los ingresos que la priori atribuye a la publicidad, draw a draw."""
    contrib = simulacion.prior["contribucion"].values.reshape(-1, len(CANALES)).sum(axis=1)
    return 100 * contrib / y.sum()


def pct_semanas_negativas(simulacion):
    """% de semanas simuladas con ingresos por debajo de cero."""
    simulado = simulacion.prior_predictive["y"].values.reshape(-1, len(df)) * y_sd + y_media
    return 100 * (simulado < 0).mean()


# (a) Qué dice la priori del taller, antes de ver un dato.
pct = pct_medios(previa)  # reutilizamos la simulación de la sección 6
print(f"contribución de los medios según la priori: mediana {np.median(pct):.0f} %, "
      f"p95 {np.percentile(pct, 95):.0f} %, p99 {np.percentile(pct, 99):.0f} %")
print(f"draws que atribuyen a la publicidad más del 100 % de los ingresos: {(pct > 100).mean():.0%}")

# (b) El "como mucho el 30 %" del jefe admite dos lecturas, y dan prioris distintas.
# La contribución es lineal en el ROI, así que multiplicar el ROI por k es desplazar mu
# de la LogNormal en log(k), y mueve todos los cuantiles a la vez.
objetivo = 30.0
mu_mediana = 0.2 + np.log(objetivo / np.median(pct))
mu_techo = 0.2 + np.log(objetivo / np.percentile(pct, 95))
print(f"\nsi el 30 % es la contribución *típica*: mu = {mu_mediana:+.2f} "
      f"(ROI mediano {np.exp(mu_mediana):.2f})")
print(f"si el 30 % es un *techo* (p95):       mu = {mu_techo:+.2f} "
      f"(ROI mediano {np.exp(mu_techo):.2f})")

previa_30 = simular_priori(mu_roi=mu_techo)
pct_30 = pct_medios(previa_30)
print(f"comprobación con la del techo: mediana {np.median(pct_30):.0f} %, "
      f"p95 {np.percentile(pct_30, 95):.0f} %")

# (c) Las prioris originales de Meridian en este modelo nacional.
previa_meridian = simular_priori(meridian=True)
print(f"\nsemanas simuladas con ingresos negativos")
print(f"  prioris del taller:  {pct_semanas_negativas(previa):.2f} %")
print(f"  prioris de Meridian: {pct_semanas_negativas(previa_meridian):.2f} %")

fig, eje = plt.subplots(figsize=(7, 3.4))
bordes = np.linspace(0, 200, 81)
eje.hist(np.clip(pct, 0, 200), bins=bordes, color="#AAAAAA", density=True,
         label=f"priori del taller (mediana {np.median(pct):.0f} %)")
eje.hist(np.clip(pct_30, 0, 200), bins=bordes, color=ACENTO, density=True, alpha=0.8,
         label=f"techo del 30 % (mediana {np.median(pct_30):.0f} %)")
eje.axvline(100, color="black", ls="--", lw=1, label="todos los ingresos")
eje.set_xlabel("% de los ingresos atribuido a los medios")
eje.legend(fontsize=9)
plt.show()

**Respuesta.** Antes de ver un solo dato, la priori atribuye a la publicidad una
contribución **mediana del 26 %** de la facturación, con p95 del 59 % y p99 del 78 %. Ningún
draw se pasa del 100 %, así que la priori no es absurda: es ancha pero posible.

Y aquí está la sorpresa del ejercicio. El jefe que dice "la publicidad explica como mucho el
30 %" no está apretando la priori: **está pidiendo una más generosa que la que teníamos**, si
se lee su 30 % como la contribución típica. Para que la mediana suba del 26 % al 30 % hay que
subir `mu` de 0,20 a 0,36, es decir, pasar de un ROI mediano de 1,22 a 1,44.

Si su 30 % es un **techo** —lo que probablemente quiere decir—, la cuenta es otra: el techo
hay que ponerlo en el p95, que ahora está en el 59 %, y eso obliga a bajar `mu` a −0,48,
o sea a un ROI mediano de 0,62 y una contribución típica del 13 %. Dicho en voz alta: *"para que tu techo del 30 % se cumpla, el
modelo tiene que empezar dando por supuesto que el canal medio pierde dinero"*. Esa frase es
el taller entero en una línea. Las restricciones que la gente da como evidentes tienen
consecuencias que nadie ha mirado, y el único sitio donde se ven es aquí, simulando.

Sobre las prioris originales de Meridian: con `sigma = HalfNormal(5)` y `gamma = Normal(0, 5)`
el **3,13 %** de las semanas simuladas sale con ingresos negativos, frente al 0,04 % de las
del taller. Son 75 veces más semanas imposibles. Con miles de observaciones repartidas por
regiones da igual, porque los datos tapan la priori; en un modelo nacional con 156 semanas,
no. Arreglarlo aquí es gratis. Después del ajuste, ya no: a esas alturas nadie va a querer
volver a tocar una priori y repetir la presentación.

## 7. Ajuste

Ahora sí, que vea los datos.

In [ ]:
RUTA_IDATA = RAIZ / "notebooks" / "idata" / "modelo_meridian.nc"

# Ponlo a False si el muestreo se te hace largo y prefieres cargar el resultado guardado.
MUESTREAR = True

if MUESTREAR:
    with modelo:
        idata = pm.sample(
            draws=1000, tune=1000, chains=4, target_accept=0.97, random_seed=SEMILLA
        )
        idata.extend(pm.sample_posterior_predictive(idata, random_seed=SEMILLA))
    RUTA_IDATA.parent.mkdir(parents=True, exist_ok=True)
    idata.to_netcdf(RUTA_IDATA, groups=["posterior", "sample_stats", "observed_data"])
else:
    idata = az.from_netcdf(RUTA_IDATA)
    with modelo:  # la predictiva posterior se recalcula, que es rápido
        idata.extend(pm.sample_posterior_predictive(idata, random_seed=SEMILLA))

## 8. Diagnóstico

Dos preguntas distintas que se confunden todo el rato:

- **¿Ha funcionado el muestreador?** `r_hat`, `ess_bulk`, `ess_tail`, divergencias.
- **¿Describe el modelo los datos?** La predictiva posterior.

Un modelo puede muestrear de maravilla y ser una tontería. Y al revés.

In [ ]:
resumen = az.summary(idata, var_names=["roi", "alpha", "ec", "gamma", "mu", "sigma"])
print("divergencias:", int(idata.sample_stats["diverging"].sum()))
print("r_hat máximo:", round(float(resumen["r_hat"].max()), 3))
print("ess_bulk mínimo:", int(resumen["ess_bulk"].min()))
resumen.round(2)

In [ ]:
az.plot_trace(idata, var_names=["roi", "sigma"], compact=True)
plt.tight_layout()
plt.show()

In [ ]:
az.plot_ppc(idata, num_pp_samples=100, colors=[ACENTO, "#7A7A7A", "black"])
plt.show()

### Ejercicio 7

- ¿Qué parámetro tiene el `ess` más bajo? ¿Te sorprende cuál es?
- Dibuja la posterior de `alpha` de cada canal contra su priori uniforme. ¿En cuáles ha
  aprendido algo el modelo y en cuáles te está devolviendo la priori con otro nombre?
- La predictiva posterior reproduce bien el centro de la distribución. Busca dónde falla.

Ojo con la conclusión fácil: que un parámetro no se mueva de su priori **no es un fallo
del muestreador**. Es información. Significa que estos datos no distinguen entre un canal
con memoria de una semana y uno con memoria de un mes, y que lo que salga por el otro lado
lo estás poniendo tú.

In [ ]:
diagnostico = az.summary(idata, var_names=["roi", "alpha", "ec", "gamma", "mu", "sigma"])
print("los cinco peores ess_bulk:")
display(diagnostico.sort_values("ess_bulk").head(5)[
    ["mean", "sd", "r_hat", "ess_bulk", "ess_tail"]
].round(2))

# Posteriori de alpha contra su priori uniforme, canal a canal.
alpha_post = idata.posterior["alpha"].values.reshape(-1, len(CANALES))
print("\ndesviación típica de la posteriori de alpha (la Uniform(0, 1) tiene 0.29):")
print(pd.Series(alpha_post.std(axis=0), index=CANALES).round(2).to_string())
fig, ejes = plt.subplots(1, len(CANALES), figsize=(13, 3), sharey=True)
for i, (eje, canal) in enumerate(zip(ejes, CANALES)):
    eje.hist(alpha_post[:, i], bins=np.linspace(0, 1, 41), density=True, color=ACENTO)
    eje.axhline(1.0, color="black", ls="--", lw=1, label="Uniform(0, 1)")
    eje.set_title(f"{canal}\nmediana {np.median(alpha_post[:, i]):.2f}", fontsize=10)
    eje.set_xlabel("alpha")
ejes[0].set_ylabel("densidad")
ejes[0].legend(fontsize=8)  # solo la uniforme lleva label
fig.tight_layout()
plt.show()

# ¿Dónde falla la predictiva posterior? En las colas, que es donde plot_ppc no se ve.
y_rep = idata.posterior_predictive["y"].values.reshape(-1, len(df)) * y_sd + y_media
cuantiles = [1, 5, 25, 50, 75, 95, 99]
display(pd.DataFrame({
    "observado (M€)": np.percentile(y, cuantiles) / 1e6,
    "predicho (M€)": np.percentile(y_rep, cuantiles) / 1e6,
}, index=[f"p{q}" for q in cuantiles]).round(2))

# Y la pregunta que el gráfico de plot_ppc no contesta: ¿acierta la semana o solo la forma?
sigma_post = float(np.median(idata.posterior["sigma"].values))
print(f"\nsigma (mediana) = {sigma_post:.2f} sobre un KPI de desviación típica 1: el modelo "
      f"explica el {100 * (1 - sigma_post ** 2):.0f} % de la varianza semanal")
print(f"el OLS del ejercicio 2 explicaba el {100 * ols.rsquared:.0f} %")

residuo = (y - y_rep.mean(axis=0)) / 1e6
print(f"autocorrelación de los residuos en el rezago 1: "
      f"{np.corrcoef(residuo[:-1], residuo[1:])[0, 1]:+.2f}")

fig, eje = plt.subplots(figsize=(10, 3))
eje.plot(df["time"], residuo, color=ACENTO)
eje.axhline(0.0, color="black", lw=0.8)
eje.set_ylabel("observado − predicho (M€)")
fig.tight_layout()
plt.show()

**Respuesta.** El peor `ess_bulk` es el de **`roi[Channel3]`**, 1.863 de 4.000 muestras, y
justo detrás va `mu`, con 1.943. Sorprende y no sorprende: `Channel3` es el canal con *más*
presupuesto, el 40 % del total. La intuición dice que el canal con más dinero es el mejor
estimado, y es al contrario, porque su contribución es tan grande que se confunde con el
nivel base: `mu` sube cuando `roi[Channel3]` baja y al revés. No es un problema del
muestreador, es que los dos parámetros explican lo mismo. Con todo, `r_hat = 1.00` y ningún
`ess` por debajo de 1.800: el muestreador ha hecho su trabajo.

Lo de `alpha` es más bonito. Las cinco posterioris tienen mediana entre 0,53 y 0,59 y
desviación típica 0,26–0,30, cuando la `Uniform(0, 1)` tiene 0,29. O sea: **el modelo no ha
aprendido nada sobre el adstock**. Lo único que hacen los datos es desaconsejar los extremos
un poco. Estas 156 semanas no distinguen un canal con memoria de una semana de uno con
memoria de un mes, y cualquier número de adstock que saques de aquí es tuyo, no de los datos.
Eso no es un fallo: es el resultado. Decirlo en voz alta es la diferencia entre un modelo y
un powerpoint.

Y el giro: la predictiva posterior no falla en ningún cuantil. p1 observado 7,10 M€ contra
7,06 predicho; p99, 9,77 contra 9,81. Clava la distribución de arriba abajo, y por eso
`plot_ppc` se ve tan bien. Pero `sigma` vale 0,86 sobre un KPI de desviación típica 1, así
que el modelo explica el **26 % de la varianza semanal**: lo mismo que el OLS del ejercicio 2,
que era un 24 %. Reproduce la *forma* de la distribución de ingresos y no acierta *qué*
semana es cada cual. Y no es que falte una estacionalidad por meter: los residuos no tienen
estructura temporal, su autocorrelación en el rezago 1 es de +0,02. Lo que hay es ruido
semanal que estos regresores no explican y que probablemente no explique ninguno.

Es el recordatorio incómodo de que la predictiva posterior es una comprobación necesaria y no
una medida de acierto. Un gráfico bonito ahí no te salva de nada.

## 9. Resultados

El gráfico que resume el taller entero: priori contra posteriori del ROI de cada canal.

In [ ]:
roi_previo = previa.prior["roi"].values.reshape(-1, len(CANALES))
roi_post = idata.posterior["roi"].values.reshape(-1, len(CANALES))

fig, ejes = plt.subplots(1, len(CANALES), figsize=(13, 3), sharey=True)
bordes = np.linspace(0, 12, 61)
for i, (eje, canal) in enumerate(zip(ejes, CANALES)):
    eje.hist(roi_previo[:, i], bins=bordes, density=True, color="#CCCCCC", label="priori")
    eje.hist(roi_post[:, i], bins=bordes, density=True, color=ACENTO, alpha=0.85,
             label="posteriori")
    eje.set_title(canal, fontsize=11)
    eje.set_xlim(0, 12)
    eje.set_xlabel("ROI")
ejes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
hdi = az.hdi(idata, var_names=["roi"], hdi_prob=0.9)["roi"].values
contrib = idata.posterior["contribucion"].values.reshape(-1, len(CANALES))

pd.DataFrame({
    "inversión (M€)": gasto_total / 1e6,
    "roi (mediana)": np.median(roi_post, axis=0),
    "roi hdi 90% bajo": hdi[:, 0],
    "roi hdi 90% alto": hdi[:, 1],
    "% ingresos (mediana)": 100 * np.median(contrib, axis=0) / y.sum(),
}, index=CANALES).round(2)

### Ejercicio 8

- Suma la contribución mediana de los cinco canales. ¿Qué porcentaje de la facturación
  atribuye el modelo a la publicidad? Compáralo con lo que decía la priori en el ejercicio
  6.
- Dibuja la **curva de respuesta** de un canal: multiplica su inversión por un factor
  entre 0 y 3, vuelve a pasar por adstock y Hill con los parámetros de la posteriori, y
  representa el ingreso incremental. Fíjate en dónde se dobla.
- ¿Coincide algún ROI con el que daba la regresión del ejercicio 2? ¿Cuál se ha movido más
  y por qué?

In [ ]:
total_post = 100 * np.median(contrib.sum(axis=1)) / y.sum()
total_previo = 100 * np.median(contrib_previa) / y.sum()
print(f"contribución de los medios: priori {total_previo:.0f} %  ->  posteriori {total_post:.0f} %")

# Curva de respuesta del canal con más presupuesto.
CANAL = int(np.argmax(gasto_total))
print(f"curva de respuesta de {CANALES[CANAL]} ({gasto_total[CANAL] / 1e6:.0f} M€ invertidos)")

n_draws = 200
sorteo = rng.choice(roi_post.shape[0], n_draws, replace=False)
alpha_d = idata.posterior["alpha"].values.reshape(-1, len(CANALES))[sorteo, CANAL]
ec_d = idata.posterior["ec"].values.reshape(-1, len(CANALES))[sorteo, CANAL]
beta_d = idata.posterior["beta"].values.reshape(-1, len(CANALES))[sorteo, CANAL]

factores = np.linspace(0.0, 3.0, 31)
pesos_d = pesos_adstock(alpha_d, MAX_LAG)  # (draws, rezagos)
curva = np.zeros((n_draws, len(factores)))
for j, k in enumerate(factores):
    rezagos_k = matriz_rezagos(x_esc[:, [CANAL]] * k, MAX_LAG)[:, :, 0]  # (semanas, rezagos)
    adstock_k = rezagos_k @ pesos_d.T                                    # (semanas, draws)
    curva[:, j] = beta_d * hill(adstock_k, ec_d).sum(axis=0) * y_sd

inversion_k = gasto_total[CANAL] * factores
fig, eje = plt.subplots(figsize=(7.5, 4))
eje.fill_between(inversion_k / 1e6, np.percentile(curva, 5, axis=0) / 1e6,
                 np.percentile(curva, 95, axis=0) / 1e6, color="#CCCCCC", label="90 %")
eje.plot(inversion_k / 1e6, np.median(curva, axis=0) / 1e6, color=ACENTO, label="mediana")
eje.plot(inversion_k / 1e6, inversion_k / 1e6, color="black", ls=":", lw=1, label="ROI = 1")
eje.axvline(gasto_total[CANAL] / 1e6, color="black", ls="--", lw=1, label="inversión actual")
eje.set_xlabel(f"inversión en {CANALES[CANAL]} en 3 años (M€)")
eje.set_ylabel("ingreso incremental (M€)")
eje.legend(fontsize=9)
fig.tight_layout()
plt.show()

for k in [0.5, 1.0, 2.0, 3.0]:
    j = int(np.argmin(np.abs(factores - k)))
    incremental = np.median(curva[:, j])
    print(f"  x{k:<4} inversión {inversion_k[j] / 1e6:5.1f} M€  ->  "
          f"incremental {incremental / 1e6:5.1f} M€  (ROI medio {incremental / inversion_k[j]:.2f})")

# OLS contra bayesiano, canal a canal.
display(pd.DataFrame({
    "roi OLS": [ols.params[f"{c}_spend"] for c in CANALES],
    "roi posteriori (mediana)": np.median(roi_post, axis=0),
    "roi priori (mediana)": np.median(roi_previo, axis=0),
}, index=CANALES).round(2))

**Respuesta.** La suma de contribuciones medianas da el **27 %** de la facturación. La
priori decía 26 %. Tres años de datos han movido el total de la contribución de medios un
punto porcentual.

Conviene sentarse un momento con eso. El reparto *entre* canales sí se ha movido —los ROI
medianos pasan de 1,2 para todos a 1,09–2,02 según el canal, y `Channel0` sube de 1,22 a
1,69—, pero el total lo fijó la priori y los datos no han tenido nada que decir. Si el número
que va a la presentación es "la publicidad nos genera el 27 % de los ingresos", ese número
lo escribiste tú en la sección 5. Esta es la pregunta que hay que hacerle a cualquier MMM de
proveedor.

La curva de respuesta de `Channel3` (88 M€ invertidos, el canal grande) se dobla pronto: a la
mitad de la inversión actual el ROI medio sería 1,70; en la inversión actual es 1,33; al
doble cae a 0,92 y al triple a 0,70. Es decir, **el último euro de este canal ya está
rindiendo por debajo de uno** en algún punto entre la inversión actual y el doble. Duplicar el
presupuesto de `Channel3` añadiría 44 M€ de ingreso incremental por 88 M€ más de inversión.
Ahí está la decisión, y no en el ROI medio.

Comparado con la regresión del ejercicio 2, todos los ROI han subido: el OLS daba 0,70–1,63
y la posteriori da 1,09–2,02. El que más se mueve es `Channel0`, de 0,77 a 1,69, y el motivo
es que el OLS le pedía explicar los ingresos con la inversión *de esa semana*, sin memoria y
sin saturación. Al añadir adstock y Hill, una misma inversión explica más. El que menos se
mueve es `Channel4`, que pasa de 0,73 a 1,09: los datos insisten en que ese canal no
funciona, y a la priori le cuesta taparlo.

## 10. La pregunta que le importa a alguien

Nadie ha pedido nunca una posteriori. Piden a dónde va el dinero del trimestre que viene.

### Ejercicio 9

Con las muestras de la posteriori, calcula:

- La probabilidad de que el canal con mejor ROI mediano sea de verdad mejor que el
  segundo: `(roi_post[:, i] > roi_post[:, j]).mean()`.
- La probabilidad de que cada canal esté perdiendo dinero (ROI < 1).
- El intervalo del 90 % del ingreso incremental de un canal, en euros.

Y luego escribe **una frase**, sin la palabra "posteriori" dentro, que puedas decirle a
quien firma el presupuesto.

In [ ]:
orden = np.argsort(np.median(roi_post, axis=0))[::-1]
mejor, segundo = int(orden[0]), int(orden[1])
probabilidad = (roi_post[:, mejor] > roi_post[:, segundo]).mean()
print(f"P({CANALES[mejor]} mejor que {CANALES[segundo]}) = {probabilidad:.0%}")

hdi_contrib = az.hdi(idata, var_names=["contribucion"], hdi_prob=0.9)["contribucion"].values
display(pd.DataFrame({
    "inversión (M€)": gasto_total / 1e6,
    "roi (mediana)": np.median(roi_post, axis=0),
    "P(ROI < 1)": (roi_post < 1).mean(axis=0),
    "incremental (M€)": np.median(contrib, axis=0) / 1e6,
    "hdi 90 % bajo": hdi_contrib[:, 0] / 1e6,
    "hdi 90 % alto": hdi_contrib[:, 1] / 1e6,
}, index=CANALES).round(2))

**Respuesta.** El canal con mejor ROI mediano es `Channel2` (2,02) y el segundo es
`Channel0` (1,69), pero la probabilidad de que `Channel2` sea de verdad mejor es del **56 %**.
Una moneda al aire. Todo el ranking de ROI que saldría en una diapositiva se sostiene en un
56 %.

Probabilidad de estar perdiendo dinero, canal a canal: `Channel4` 45 %, `Channel1` 37 %,
`Channel3` 31 %, `Channel2` 26 %, `Channel0` 25 %. Ninguno está a salvo y ninguno está
descartado. Y el intervalo del 90 % del ingreso incremental de `Channel3`, el canal en el que
se va el 40 % del presupuesto: entre **16 y 228 M€** sobre 88 M€ invertidos. Eso es lo que
hay, y es mucho menos de lo que parecía cuando solo mirábamos la mediana.

La frase para quien firma el presupuesto:

> *Con lo que hemos invertido estos tres años no podemos distinguir qué canal funciona mejor:
> el que mejor pinta le saca al segundo lo que una moneda al aire. De lo que sí estamos
> razonablemente seguros es de que Channel4 es el peor candidato —casi una de cada dos
> posibilidades de no recuperar lo invertido—, así que si hay que mover dinero, se mueve de
> ahí. Y si quieres una respuesta mejor el año que viene, hay que apagar un canal en unas
> cuantas regiones durante unas semanas, porque esa medición no sale de los históricos.*

Eso último no es una coletilla. Meridian existe para combinar el histórico con experimentos;
el histórico solo da para esto.

## Para llevarte a casa

- El modelo no ha descubierto los ROI: ha combinado unos datos flojos con una priori que
  has elegido tú. Cambia la priori y cambian los resultados. Eso no es un defecto del
  método bayesiano, es lo que el método te obliga a enseñar.
- La pregunta que hay que hacerle a cualquier MMM, propio o de proveedor, es la del
  ejercicio 6: **¿qué contribución de medios asume tu priori antes de ver los datos?** Si
  no saben responder, el número que te van a dar es el suyo, no el de los datos.
- Lo que dejamos fuera: el canal orgánico, la variable `Promo`, la estacionalidad con
  varios *knots*, la calibración del ROI con experimentos y —claro— el modelo jerárquico
  por regiones, que es donde de verdad brilla Meridian.